# Streaming Attack Detection with Sketches

**Improvements over v1:**
- GPU runtime: `%pip install cupy-cuda12x` with fallback
- All plots use **Plotly** (renders natively in Databricks)
- Attack window threshold lowered from >50% to >5% (fixes near-zero attack windows)
- `class_weight='balanced'` on RandomForest (fixes recall ~0.18)
- GradientBoosting classifier added
- Probability-threshold tuning via Precision-Recall curve
- Multi-window-size comparison (10s / 30s / 60s)
- 3 new derived features: port-scan ratio, src/dst IP ratio, byte asymmetry

1. Efi Pecani
2. Guy Hasson

In [0]:
%pip install cupy-cuda12x plotly kaleido
# dbutils.library.restartPython()

  Obtaining dependency information for cupy-cuda12x from https://files.pythonhosted.org/packages/63/12/a59be7517c2c8f05a31cbf4a84597b8d686050acf99e77b1f95fcc0da327/cupy_cuda12x-14.0.0-cp311-cp311-manylinux2014_x86_64.whl.metadata
  Using cached cupy_cuda12x-14.0.0-cp311-cp311-manylinux2014_x86_64.whl.metadata (2.8 kB)
  Obtaining dependency information for kaleido from https://files.pythonhosted.org/packages/4b/97/f6de8d4af54d6401d6581a686cce3e3e2371a79ba459a449104e026c08bc/kaleido-1.2.0-py3-none-any.whl.metadata
  Using cached kaleido-1.2.0-py3-none-any.whl.metadata (5.6 kB)
  Obtaining dependency information for numpy<2.6,>=2.0 from https://files.pythonhosted.org/packages/1b/46/6fa4ea94f1ddf969b2ee941290cca6f1bfac92b53c76ae5f44afe17ceb69/numpy-2.4.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Using cached numpy-2.4.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Obtaining dependency information for cuda-pathfinder==1.*,>=1.

In [0]:
import numpy as np
import pandas as pd
import time
import os
import warnings
warnings.filterwarnings("ignore")

# --------------- GPU auto-detection ---------------
# CuPy can import fine but fail on actual GPU ops if CUDA libs are mismatched.
# We run a small smoke test to catch that early.
GPU_AVAILABLE = False
try:
    import cupy as cp
    # Smoke test: force a real kernel launch to verify CUDA runtime works
    _test = cp.array([1.0, 2.0])
    _ = float(cp.sum(_test))
    del _test
    xp = cp
    GPU_AVAILABLE = True
except Exception as _gpu_err:
    print(f"CuPy GPU unavailable ({type(_gpu_err).__name__}: {_gpu_err}), falling back to NumPy.")
    xp = np

def to_device(arr):
    """Move numpy array to the compute device (GPU if available)."""
    if GPU_AVAILABLE:
        return cp.asarray(arr)
    return np.asarray(arr)

def to_host(arr):
    """Move array back to CPU numpy."""
    if GPU_AVAILABLE and hasattr(arr, "get"):
        return cp.asnumpy(arr)
    return np.asarray(arr)

# --------------- cuML auto-detection ---------------
try:
    from cuml.ensemble import RandomForestClassifier as cuRFC
    USE_CUML = True
except ImportError:
    USE_CUML = False

from sklearn.ensemble import RandomForestClassifier as skRFC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, auc,
)
from sklearn.preprocessing import StandardScaler

# --------------- Plotly ---------------
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --------------- Configuration ---------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = "UNSW_NB15"
CSV_FILES = [os.path.join(DATA_DIR, f"UNSW-NB15_{i}.csv") for i in range(1, 5)]

COLUMN_NAMES = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb",
    "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "Sjit", "Djit",
    "Stime", "Ltime", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
    "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
    "attack_cat", "label",
]

WINDOW_SIZES = [10, 30, 60]  # seconds — assignment requires comparison
ATTACK_THRESHOLD = 0.05      # label window as attack if >5% flows are attacks

print(f"GPU available (CuPy): {GPU_AVAILABLE}")
print(f"cuML available:       {USE_CUML}")
print(f"Array backend (xp):   {'cupy' if GPU_AVAILABLE else 'numpy'}")
print(f"Window sizes: {WINDOW_SIZES}s")
print(f"Attack threshold: {ATTACK_THRESHOLD}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Data files: {CSV_FILES}")

CuPy GPU unavailable (ImportError: 
Failed to import CuPy.

If you installed CuPy via wheels (cupy-cudaXXX or cupy-rocm-X-X), make sure that the package matches with the version of CUDA or ROCm installed.

On Linux, you may need to set LD_LIBRARY_PATH environment variable depending on how you installed CUDA/ROCm.
On Windows, try setting CUDA_PATH environment variable.

Check the Installation Guide for details:
  https://docs.cupy.dev/en/latest/install.html

Original error:
  ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).
), falling back to NumPy.


Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0x7fad8f797240>
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
                   ^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
             ^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'split'


GPU available (CuPy): False
cuML available:       False
Array backend (xp):   numpy
Window sizes: [10, 30, 60]s
Attack threshold: 0.05
Random seed: 42
Data files: ['UNSW_NB15/UNSW-NB15_1.csv', 'UNSW_NB15/UNSW-NB15_2.csv', 'UNSW_NB15/UNSW-NB15_3.csv', 'UNSW_NB15/UNSW-NB15_4.csv']


In [0]:
# Quick data exploration
sample = pd.read_csv(CSV_FILES[0], header=None, names=COLUMN_NAMES, nrows=5)
print("Sample records from UNSW-NB15_1.csv:")
print(
    sample[
        ["srcip", "sport", "dstip", "dsport", "proto", "Stime",
         "sbytes", "dbytes", "attack_cat", "label"]
    ].to_string()
)
print(f"\nStime range check (first file, first 1000 rows):")
sample_1k = pd.read_csv(CSV_FILES[0], header=None, names=COLUMN_NAMES, nrows=1000)
print(f"  Min Stime: {sample_1k['Stime'].min()} -> {pd.to_datetime(sample_1k['Stime'].min(), unit='s')}")
print(f"  Max Stime: {sample_1k['Stime'].max()} -> {pd.to_datetime(sample_1k['Stime'].max(), unit='s')}")
print(f"  Label distribution: {sample_1k['label'].value_counts().to_dict()}")

Sample records from UNSW-NB15_1.csv:
        srcip  sport          dstip  dsport proto       Stime  sbytes  dbytes  attack_cat  label
0  59.166.0.0   1390  149.171.126.6      53   udp  1421927414     132     164         NaN      0
1  59.166.0.0  33661  149.171.126.9    1024   udp  1421927414     528     304         NaN      0
2  59.166.0.6   1464  149.171.126.7      53   udp  1421927414     146     178         NaN      0
3  59.166.0.5   3593  149.171.126.5      53   udp  1421927414     132     164         NaN      0
4  59.166.0.3  49664  149.171.126.0      53   udp  1421927414     146     178         NaN      0

Stime range check (first file, first 1000 rows):
  Min Stime: 1421927377 -> 2015-01-22 11:49:37
  Max Stime: 1421927449 -> 2015-01-22 11:50:49
  Label distribution: {0: 965, 1: 35}


In [0]:
# Count total records and compute the 70/30 split point
total_records = 0
for f in CSV_FILES:
    count = sum(1 for _ in open(f, "r", encoding="utf-8", errors="replace"))
    print(f"{f}: {count} records")
    total_records += count

print(f"\nTotal records: {total_records}")
TRAIN_COUNT = int(total_records * 0.7)
TEST_COUNT = total_records - TRAIN_COUNT
print(f"Train: first {TRAIN_COUNT} records (70%)")
print(f"Test: last {TEST_COUNT} records (30%)")

UNSW_NB15/UNSW-NB15_1.csv: 700001 records
UNSW_NB15/UNSW-NB15_2.csv: 700001 records
UNSW_NB15/UNSW-NB15_3.csv: 700001 records
UNSW_NB15/UNSW-NB15_4.csv: 440044 records

Total records: 2540047
Train: first 1778032 records (70%)
Test: last 762015 records (30%)


## Part 2: Vectorized Hash Utilities

We replace per-element MD5 with a **multiply-shift** integer hash family:

$$h_{a,b}(x) = (a \cdot x + b) \bmod 2^{64}$$

where $a$ is odd. This is 2-universal, sufficient for all our sketches, and fully vectorisable
with numpy/cupy broadcasting: one call hashes N keys with K seeds simultaneously.

In [0]:
# ---- Vectorized hash primitives ------------------------------------------------

def _gen_hash_params(n, rng):
    """Generate (a, b) parameter arrays for *n* multiply-shift hash functions."""
    a_raw = rng.randint(0, 2**62, size=n)
    b_raw = rng.randint(0, 2**62, size=n)
    a = (a_raw.astype(np.uint64) << np.uint64(1)) | np.uint64(1)  # always odd
    b = b_raw.astype(np.uint64)
    return a, b


def hash_batch(keys, a, b):
    """
    Multiply-shift hash (vectorized).

    Parameters
    ----------
    keys : (N,) uint64 array on device
    a, b : (K,) uint64 arrays on device

    Returns
    -------
    (K, N) uint64 array of raw hash values (mod 2^64 via overflow).
    """
    return a[:, None] * keys[None, :] + b[:, None]


def hash_to_buckets(raw_hashes, num_buckets):
    """Map raw uint64 hashes to bucket indices in [0, num_buckets).

    Uses the upper 32 bits for best uniformity with multiply-shift.
    """
    return ((raw_hashes >> np.uint64(32)) % np.uint64(num_buckets)).astype(np.int64)


def hash_to_signs(raw_hashes):
    """Map raw uint64 hashes to signs in {-1, +1} using the top bit."""
    return np.int64(1) - np.int64(2) * (raw_hashes >> np.uint64(63)).astype(np.int64)


# ---- IP / port conversion helpers -----------------------------------------------

def ips_to_uint64(ip_col):
    """Convert IP address column to uint64 array (vectorized)."""
    s = ip_col.astype(str)
    parts = s.str.split(".", expand=True)
    if parts.shape[1] >= 4:
        octets = parts.iloc[:, :4].apply(pd.to_numeric, errors="coerce").fillna(0)
        v = octets.values.astype(np.uint64)
        return (v[:, 0] << np.uint64(24)) | (v[:, 1] << np.uint64(16)) | (v[:, 2] << np.uint64(8)) | v[:, 3]
    # Fallback for non-dotted-decimal formats
    return pd.util.hash_pandas_object(s, index=False).values.view(np.uint64)


def ports_to_uint64(port_col):
    """Convert port column to uint64 array."""
    return pd.to_numeric(port_col, errors="coerce").fillna(0).values.astype(np.uint64)


print("Vectorized hash utilities ready.")

Vectorized hash utilities ready.


## Part 3: Sketch Implementations

### Median-of-Means Aggregation

All frequency moment estimators use the **median-of-means** trick:
given R estimates arranged as `num_groups x group_size`,
compute the mean within each group and return the median of the group means.

### Vectorized Batch Updates

Each sketch exposes an `update_batch(keys)` method that processes an entire
numpy / cupy array at once — replacing the inner Python loop over individual events.

In [0]:
def median_of_means(estimates, num_groups, group_size):
    """Median-of-means aggregation over a 1-D array of estimates."""
    arr = np.asarray(estimates, dtype=np.float64).reshape(num_groups, group_size)
    return float(np.median(np.mean(arr, axis=1)))


# =============================================================================
# F1 Estimator — Morris approximate counter (vectorized across R counters)
# =============================================================================
class MorrisCounter:
    """
    Morris approximate counter for F1 (total count / volume).

    R independent copies with median-of-means give a reliable estimate.
    The counter is inherently sequential across events but vectorized across R.
    """

    def __init__(self, R=64, num_groups=8, seed_offset=0):
        self.R = R
        self.num_groups = num_groups
        self.group_size = R // num_groups
        self.counters = np.zeros(R, dtype=np.float64)  # always on CPU
        self._so = seed_offset
        self.rng = np.random.RandomState(RANDOM_SEED + seed_offset)

    def update_batch(self, n_events):
        """Process *n_events* events (vectorized across R counters per event)."""
        for _ in range(n_events):
            rands = self.rng.random(self.R)
            probs = np.power(2.0, -self.counters)
            self.counters += (rands < probs).astype(np.float64)

    def estimate(self):
        individual = np.power(2.0, self.counters) - 1.0
        return median_of_means(individual, self.num_groups, self.group_size)

    def memory_bytes(self):
        return self.R * 8

    def reset(self):
        self.counters[:] = 0
        self.rng = np.random.RandomState(RANDOM_SEED + self._so)


# =============================================================================
# F0 Estimator — Flajolet-Martin with leading zeros (vectorized)
# =============================================================================
class FMEstimator:
    """
    Flajolet-Martin style F0 estimator using **leading zeros** of
    multiply-shift hashes (upper bits are well-distributed).

    `update_batch` hashes an entire key array at once.
    """

    def __init__(self, R=64, num_groups=8, seed_offset=100):
        self.R = R
        self.num_groups = num_groups
        self.group_size = R // num_groups
        self.max_zeros = xp.zeros(R, dtype=xp.int32)
        rng = np.random.RandomState(RANDOM_SEED + seed_offset)
        a, b = _gen_hash_params(R, rng)
        self.a = to_device(a)
        self.b = to_device(b)
        self._so = seed_offset

    def update_batch(self, keys):
        """keys: (N,) uint64 array on device."""
        if len(keys) == 0:
            return
        raw = hash_batch(keys, self.a, self.b)  # (R, N)
        safe = xp.maximum(raw, xp.uint64(1))
        lz = 63 - xp.floor(xp.log2(safe.astype(xp.float64))).astype(xp.int32)
        self.max_zeros = xp.maximum(self.max_zeros, xp.max(lz, axis=1))

    def estimate(self):
        mz = to_host(self.max_zeros).astype(np.float64)
        individual = np.power(2.0, mz)
        return median_of_means(individual, self.num_groups, self.group_size)

    def memory_bytes(self):
        return self.R * 4

    def reset(self):
        self.max_zeros = xp.zeros(self.R, dtype=xp.int32)


# =============================================================================
# F2 Estimator — AMS (vectorized)
# =============================================================================
class AMSEstimator:
    """
    AMS estimator for F2 (second frequency moment).

    For each repetition j, Z_j += s_j(x). Estimate = Z_j^2.
    `update_batch` computes signs for all keys x all repetitions in one broadcast.
    """

    def __init__(self, R=64, num_groups=8, seed_offset=200):
        self.R = R
        self.num_groups = num_groups
        self.group_size = R // num_groups
        self.Z = xp.zeros(R, dtype=xp.float64)
        rng = np.random.RandomState(RANDOM_SEED + seed_offset)
        a, b = _gen_hash_params(R, rng)
        self.a = to_device(a)
        self.b = to_device(b)
        self._so = seed_offset

    def update_batch(self, keys):
        if len(keys) == 0:
            return
        raw = hash_batch(keys, self.a, self.b)  # (R, N)
        signs = hash_to_signs(raw)               # (R, N) in {-1,+1}
        self.Z += xp.sum(signs.astype(xp.float64), axis=1)

    def estimate(self):
        z = to_host(self.Z)
        individual = z ** 2
        return median_of_means(individual, self.num_groups, self.group_size)

    def memory_bytes(self):
        return self.R * 8

    def reset(self):
        self.Z = xp.zeros(self.R, dtype=xp.float64)


# =============================================================================
# Count-Min Sketch (vectorized scatter-add via bincount)
# =============================================================================
class CountMinSketch:
    """
    Count-Min Sketch (d rows x w columns).

    `update_batch` hashes all keys at once and uses `bincount` for scatter-add.
    `query_batch` returns min-across-rows estimates for a batch of keys.
    """

    def __init__(self, d=4, w=2048, seed_offset=300):
        self.d = d
        self.w = w
        self.table = xp.zeros((d, w), dtype=xp.int64)
        rng = np.random.RandomState(RANDOM_SEED + seed_offset)
        a, b = _gen_hash_params(d, rng)
        self.a = to_device(a)
        self.b = to_device(b)

    def update_batch(self, keys):
        if len(keys) == 0:
            return
        raw = hash_batch(keys, self.a, self.b)           # (d, N)
        buckets = hash_to_buckets(raw, self.w)            # (d, N) int64
        for i in range(self.d):
            bc = xp.bincount(buckets[i], minlength=self.w)
            self.table[i] += bc[: self.w]

    def query_batch(self, keys):
        """Return (N,) int64 array of point-query estimates (min across rows)."""
        if len(keys) == 0:
            return xp.array([], dtype=xp.int64)
        raw = hash_batch(keys, self.a, self.b)
        buckets = hash_to_buckets(raw, self.w)
        rows = xp.stack([self.table[i, buckets[i]] for i in range(self.d)])
        return xp.min(rows, axis=0)

    def memory_bytes(self):
        return self.d * self.w * 8

    def reset(self):
        self.table = xp.zeros((self.d, self.w), dtype=xp.int64)


# =============================================================================
# Count-Sketch (vectorized scatter-add with signs)
# =============================================================================
class CountSketch:
    """
    Count-Sketch (r rows x b buckets) — unbiased frequency estimator.

    Uses separate bucket and sign hash families.
    """

    def __init__(self, r=5, b=2048, seed_offset=400):
        self.r = r
        self.b_size = b
        self.table = xp.zeros((r, b), dtype=xp.int64)
        rng = np.random.RandomState(RANDOM_SEED + seed_offset)
        ba, bb = _gen_hash_params(r, rng)
        sa, sb = _gen_hash_params(r, rng)
        self.bucket_a = to_device(ba)
        self.bucket_b = to_device(bb)
        self.sign_a = to_device(sa)
        self.sign_b = to_device(sb)

    def update_batch(self, keys):
        if len(keys) == 0:
            return
        raw_bkt = hash_batch(keys, self.bucket_a, self.bucket_b)
        raw_sgn = hash_batch(keys, self.sign_a, self.sign_b)
        buckets = hash_to_buckets(raw_bkt, self.b_size)
        signs = hash_to_signs(raw_sgn)
        for i in range(self.r):
            w = signs[i].astype(xp.float64)
            bc = xp.bincount(buckets[i], weights=w, minlength=self.b_size)
            self.table[i] += bc[: self.b_size].astype(xp.int64)

    def query_batch(self, keys):
        """Return (N,) float64 array — median of signed estimates per key."""
        if len(keys) == 0:
            return xp.array([], dtype=xp.float64)
        raw_bkt = hash_batch(keys, self.bucket_a, self.bucket_b)
        raw_sgn = hash_batch(keys, self.sign_a, self.sign_b)
        buckets = hash_to_buckets(raw_bkt, self.b_size)
        signs = hash_to_signs(raw_sgn)
        rows = xp.stack([self.table[i, buckets[i]] * signs[i] for i in range(self.r)])
        return xp.median(rows.astype(xp.float64), axis=0)

    def memory_bytes(self):
        return self.r * self.b_size * 8

    def reset(self):
        self.table = xp.zeros((self.r, self.b_size), dtype=xp.int64)


print("All 5 sketch data structures implemented (vectorized):")
print("  - MorrisCounter  (F1)")
print("  - FMEstimator    (F0)")
print("  - AMSEstimator   (F2)")
print("  - CountMinSketch (CMS)")
print("  - CountSketch    (CS)")

All 5 sketch data structures implemented (vectorized):
  - MorrisCounter  (F1)
  - FMEstimator    (F0)
  - AMSEstimator   (F2)
  - CountMinSketch (CMS)
  - CountSketch    (CS)


In [0]:
# === Sketch sanity tests ===
print("=== Sketch Sanity Tests ===\n")

# F1 (Morris) — count 1000 events
mc = MorrisCounter(R=64, num_groups=8, seed_offset=0)
mc.update_batch(1000)
f1_est = mc.estimate()
print(f"F1 (Morris): true=1000, est={f1_est:.1f}, err={abs(f1_est-1000)/1000:.2%}")

# F0 (FM) — 200 distinct elements, 1000 total
fm = FMEstimator(R=64, num_groups=8, seed_offset=100)
keys_f0 = to_device(np.array([i % 200 for i in range(1000)], dtype=np.uint64))
fm.update_batch(keys_f0)
f0_est = fm.estimate()
print(f"F0 (FM):     true=200,  est={f0_est:.1f}, err={abs(f0_est-200)/200:.2%}")

# F2 (AMS) — 100xA + 50xB + 20xC -> true F2=12900
ams = AMSEstimator(R=64, num_groups=8, seed_offset=200)
keys_f2 = to_device(np.concatenate([
    np.full(100, 1, dtype=np.uint64),
    np.full(50, 2, dtype=np.uint64),
    np.full(20, 3, dtype=np.uint64),
]))
ams.update_batch(keys_f2)
f2_est = ams.estimate()
print(f"F2 (AMS):    true=12900, est={f2_est:.1f}, err={abs(f2_est-12900)/12900:.2%}")

# CMS — point queries
cms = CountMinSketch(d=4, w=2048, seed_offset=300)
cms_keys = to_device(np.concatenate([
    np.full(100, 10, dtype=np.uint64),
    np.full(10, 20, dtype=np.uint64),
]))
cms.update_batch(cms_keys)
q = to_host(cms.query_batch(to_device(np.array([10, 20], dtype=np.uint64))))
print(f"CMS:         query(10)={q[0]} (true=100), query(20)={q[1]} (true=10)")

# Count-Sketch — point queries
cs = CountSketch(r=5, b=2048, seed_offset=400)
cs.update_batch(cms_keys)
q2 = to_host(cs.query_batch(to_device(np.array([10, 20], dtype=np.uint64))))
print(f"CS:          query(10)={q2[0]:.0f} (true=100), query(20)={q2[1]:.0f} (true=10)")

print("\nAll sketches pass sanity checks.")

=== Sketch Sanity Tests ===

F1 (Morris): true=1000, est=879.0, err=12.10%
F0 (FM):     true=200,  est=360.5, err=80.25%
F2 (AMS):    true=12900, est=18150.0, err=40.70%
CMS:         query(10)=100 (true=100), query(20)=10 (true=10)
CS:          query(10)=100 (true=100), query(20)=10 (true=10)

All sketches pass sanity checks.


## Part 4: Streaming Engine and Feature Extraction

### Window Processor

For each tumbling window we maintain sketches for F0, F1, F2, CMS, and CS.
The `update_batch` method feeds an entire window's worth of events at once.

### Feature Vector (13 features per window)

| # | Feature | Source | Description |
|---|---------|--------|-------------|
| 1 | `f0_srcip` | FM | Distinct source IPs |
| 2 | `f0_dstip` | FM | Distinct destination IPs |
| 3 | `f0_dstport` | FM | Distinct destination ports |
| 4 | `f1_count` | Morris | Total flow count |
| 5 | `f2_srcip` | AMS | F2 of srcIP distribution |
| 6 | `f2_dstip` | AMS | F2 of dstIP distribution |
| 7 | `f2_srcip_norm` | AMS/Morris | Normalized burst F2/F1^2 |
| 8 | `cms_max_srcip` | CMS | Max estimated srcIP count |
| 9 | `cms_topk_frac` | CMS | Fraction of traffic in top-5 srcIPs |
| 10 | `cs_max_srcip` | CS | Count-Sketch max srcIP estimate |
| 11 | `f0_dstport_per_srcip` | FM | Port-scan indicator (distinct ports / distinct srcIPs) |
| 12 | `srcip_to_dstip_ratio` | FM | Many-to-one / DDoS indicator |
| 13 | `byte_ratio` | Accumulator | Traffic asymmetry sbytes / (sbytes + dbytes) |

In [0]:
FEATURE_NAMES = [
    "f0_srcip", "f0_dstip", "f0_dstport",
    "f1_count",
    "f2_srcip", "f2_dstip", "f2_srcip_norm",
    "cms_max_srcip", "cms_topk_frac",
    "cs_max_srcip",
    # --- 3 new derived features ---
    "f0_dstport_per_srcip",
    "srcip_to_dstip_ratio",
    "byte_ratio",
]
TOP_K = 5


class SketchWindowProcessor:
    """Manages all sketches for a single tumbling window (batch interface)."""

    def __init__(self, cms_d=4, cms_w=2048, cs_r=5, cs_b=2048,
                 attack_threshold=ATTACK_THRESHOLD):
        self.f0_srcip = FMEstimator(R=64, num_groups=8, seed_offset=100)
        self.f0_dstip = FMEstimator(R=64, num_groups=8, seed_offset=170)
        self.f0_dstport = FMEstimator(R=64, num_groups=8, seed_offset=240)
        self.f1_counter = MorrisCounter(R=64, num_groups=8, seed_offset=0)
        self.f2_srcip = AMSEstimator(R=64, num_groups=8, seed_offset=200)
        self.f2_dstip = AMSEstimator(R=64, num_groups=8, seed_offset=270)
        self.cms = CountMinSketch(d=cms_d, w=cms_w, seed_offset=300)
        self.cs = CountSketch(r=cs_r, b=cs_b, seed_offset=400)
        self.candidate_keys = xp.array([], dtype=xp.uint64)
        self.max_candidates = TOP_K * 4
        self.event_count = 0
        self.attack_count = 0
        self.sbytes_sum = 0.0
        self.dbytes_sum = 0.0
        self.attack_threshold = attack_threshold
        self.cms_d = cms_d
        self.cms_w = cms_w
        self.cs_r = cs_r
        self.cs_b = cs_b

    def update_batch(self, srcip_keys, dstip_keys, dstport_keys, labels,
                     sbytes_arr=None, dbytes_arr=None):
        """
        Process a batch of events (one window or partial window).

        All key arrays are uint64 on compute device.
        labels is a numpy int32 array on CPU.
        sbytes_arr, dbytes_arr are optional numpy float64 arrays on CPU.
        """
        n = len(srcip_keys)
        if n == 0:
            return
        self.event_count += n
        self.attack_count += int(np.sum(labels))

        # Byte accumulators (simple int sums, O(1) memory)
        if sbytes_arr is not None:
            self.sbytes_sum += float(np.sum(sbytes_arr))
        if dbytes_arr is not None:
            self.dbytes_sum += float(np.sum(dbytes_arr))

        # F0 estimators
        self.f0_srcip.update_batch(srcip_keys)
        self.f0_dstip.update_batch(dstip_keys)
        self.f0_dstport.update_batch(dstport_keys)

        # F1 (sequential across events, vectorized across R)
        self.f1_counter.update_batch(n)

        # F2 estimators
        self.f2_srcip.update_batch(srcip_keys)
        self.f2_dstip.update_batch(dstip_keys)

        # CMS and Count-Sketch (srcIP only)
        self.cms.update_batch(srcip_keys)
        self.cs.update_batch(srcip_keys)

        # Maintain candidate heavy-hitter keys for later query
        unique_src = xp.unique(srcip_keys)
        if len(self.candidate_keys) > 0:
            self.candidate_keys = xp.unique(
                xp.concatenate([self.candidate_keys, unique_src])
            )
        else:
            self.candidate_keys = unique_src

        # Prune candidates if too many
        if len(self.candidate_keys) > self.max_candidates:
            ests = self.cms.query_batch(self.candidate_keys)
            top_idx = xp.argsort(ests)[-(TOP_K * 2):]
            self.candidate_keys = self.candidate_keys[top_idx]

    def finalize(self):
        """Extract the 13-feature vector and the window label."""
        f0_src = self.f0_srcip.estimate()
        f0_dst = self.f0_dstip.estimate()
        f0_port = self.f0_dstport.estimate()
        f1 = self.f1_counter.estimate()
        f2_src = self.f2_srcip.estimate()
        f2_dst = self.f2_dstip.estimate()

        f1_safe = max(f1, 1.0)
        f2_norm = f2_src / (f1_safe ** 2)

        if len(self.candidate_keys) > 0:
            cms_ests = to_host(self.cms.query_batch(self.candidate_keys))
            cs_ests = to_host(self.cs.query_batch(self.candidate_keys))
            top_k_idx = np.argsort(cms_ests)[-TOP_K:]
            cms_max = float(np.max(cms_ests))
            cms_topk_sum = float(np.sum(cms_ests[top_k_idx]))
            cms_topk_frac = cms_topk_sum / f1_safe
            cs_max = float(np.max(cs_ests))
        else:
            cms_max = 0.0
            cms_topk_frac = 0.0
            cs_max = 0.0

        # Derived features
        f0_src_safe = max(f0_src, 1.0)
        f0_dst_safe = max(f0_dst, 1.0)
        f0_dstport_per_srcip = f0_port / f0_src_safe
        srcip_to_dstip_ratio = f0_src / f0_dst_safe
        total_bytes = self.sbytes_sum + self.dbytes_sum
        byte_ratio = self.sbytes_sum / total_bytes if total_bytes > 0 else 0.5

        features = {
            "f0_srcip": f0_src,
            "f0_dstip": f0_dst,
            "f0_dstport": f0_port,
            "f1_count": f1,
            "f2_srcip": f2_src,
            "f2_dstip": f2_dst,
            "f2_srcip_norm": f2_norm,
            "cms_max_srcip": cms_max,
            "cms_topk_frac": cms_topk_frac,
            "cs_max_srcip": cs_max,
            "f0_dstport_per_srcip": f0_dstport_per_srcip,
            "srcip_to_dstip_ratio": srcip_to_dstip_ratio,
            "byte_ratio": byte_ratio,
        }

        # FIX: Use configurable threshold instead of majority vote
        label = 1 if (self.attack_count / max(self.event_count, 1)) > self.attack_threshold else 0
        return features, label, self.event_count

    def memory_bytes(self):
        mem = (
            self.f0_srcip.memory_bytes()
            + self.f0_dstip.memory_bytes()
            + self.f0_dstport.memory_bytes()
            + self.f1_counter.memory_bytes()
            + self.f2_srcip.memory_bytes()
            + self.f2_dstip.memory_bytes()
            + self.cms.memory_bytes()
            + self.cs.memory_bytes()
        )
        mem += len(self.candidate_keys) * 8
        mem += 16  # sbytes_sum + dbytes_sum accumulators
        return mem

    def reset(self):
        self.f0_srcip.reset()
        self.f0_dstip.reset()
        self.f0_dstport.reset()
        self.f1_counter.reset()
        self.f2_srcip.reset()
        self.f2_dstip.reset()
        self.cms.reset()
        self.cs.reset()
        self.candidate_keys = xp.array([], dtype=xp.uint64)
        self.event_count = 0
        self.attack_count = 0
        self.sbytes_sum = 0.0
        self.dbytes_sum = 0.0


class ExactWindowProcessor:
    """Exact baseline — uses dicts / sets instead of sketches."""

    def __init__(self, attack_threshold=ATTACK_THRESHOLD):
        self.srcip_freq = {}
        self.dstip_freq = {}
        self.srcip_set = set()
        self.dstip_set = set()
        self.dstport_set = set()
        self.event_count = 0
        self.attack_count = 0
        self.sbytes_sum = 0.0
        self.dbytes_sum = 0.0
        self.attack_threshold = attack_threshold

    def update_batch(self, srcips, dstips, dstports, labels,
                     sbytes_arr=None, dbytes_arr=None):
        """numpy uint64 arrays on CPU."""
        n = len(srcips)
        self.event_count += n
        self.attack_count += int(np.sum(labels))

        if sbytes_arr is not None:
            self.sbytes_sum += float(np.sum(sbytes_arr))
        if dbytes_arr is not None:
            self.dbytes_sum += float(np.sum(dbytes_arr))

        u_src, c_src = np.unique(srcips, return_counts=True)
        for ip, c in zip(u_src, c_src):
            k = int(ip)
            self.srcip_set.add(k)
            self.srcip_freq[k] = self.srcip_freq.get(k, 0) + int(c)

        u_dst, c_dst = np.unique(dstips, return_counts=True)
        for ip, c in zip(u_dst, c_dst):
            k = int(ip)
            self.dstip_set.add(k)
            self.dstip_freq[k] = self.dstip_freq.get(k, 0) + int(c)

        self.dstport_set.update(int(p) for p in np.unique(dstports))

    def finalize(self):
        f0_src = len(self.srcip_set)
        f0_dst = len(self.dstip_set)
        f0_port = len(self.dstport_set)
        f1 = self.event_count
        f2_src = sum(v ** 2 for v in self.srcip_freq.values())
        f2_dst = sum(v ** 2 for v in self.dstip_freq.values())
        f1_safe = max(f1, 1)
        f2_norm = f2_src / (f1_safe ** 2)

        sorted_src = sorted(self.srcip_freq.items(), key=lambda x: x[1], reverse=True)
        top_k = sorted_src[:TOP_K]
        cms_max = top_k[0][1] if top_k else 0
        topk_sum = sum(c for _, c in top_k)
        cms_topk_frac = topk_sum / f1_safe
        cs_max = cms_max  # exact = same value

        # Derived features
        f0_src_safe = max(f0_src, 1)
        f0_dst_safe = max(f0_dst, 1)
        f0_dstport_per_srcip = f0_port / f0_src_safe
        srcip_to_dstip_ratio = f0_src / f0_dst_safe
        total_bytes = self.sbytes_sum + self.dbytes_sum
        byte_ratio = self.sbytes_sum / total_bytes if total_bytes > 0 else 0.5

        features = {
            "f0_srcip": f0_src,
            "f0_dstip": f0_dst,
            "f0_dstport": f0_port,
            "f1_count": f1,
            "f2_srcip": f2_src,
            "f2_dstip": f2_dst,
            "f2_srcip_norm": f2_norm,
            "cms_max_srcip": cms_max,
            "cms_topk_frac": cms_topk_frac,
            "cs_max_srcip": cs_max,
            "f0_dstport_per_srcip": f0_dstport_per_srcip,
            "srcip_to_dstip_ratio": srcip_to_dstip_ratio,
            "byte_ratio": byte_ratio,
        }

        # FIX: configurable threshold
        label = 1 if (self.attack_count / max(self.event_count, 1)) > self.attack_threshold else 0
        return features, label, self.event_count

    def reset(self):
        self.srcip_freq.clear()
        self.dstip_freq.clear()
        self.srcip_set.clear()
        self.dstip_set.clear()
        self.dstport_set.clear()
        self.event_count = 0
        self.attack_count = 0
        self.sbytes_sum = 0.0
        self.dbytes_sum = 0.0


print("Window processors ready (Sketch-based and Exact).")
print(f"Feature vector ({len(FEATURE_NAMES)} features): {FEATURE_NAMES}")
print(f"Sketch window memory: ~{SketchWindowProcessor().memory_bytes()/1024:.1f} KB")

Window processors ready (Sketch-based and Exact).
Feature vector (13 features): ['f0_srcip', 'f0_dstip', 'f0_dstport', 'f1_count', 'f2_srcip', 'f2_dstip', 'f2_srcip_norm', 'cms_max_srcip', 'cms_topk_frac', 'cs_max_srcip', 'f0_dstport_per_srcip', 'srcip_to_dstip_ratio', 'byte_ratio']
Sketch window memory: ~146.3 KB


In [0]:
def stream_process(
    csv_files,
    column_names,
    window_size,
    train_count,
    cms_d=4,
    cms_w=2048,
    cs_r=5,
    cs_b=2048,
    attack_threshold=ATTACK_THRESHOLD,
    run_exact=True,
    chunk_size=50000,
):
    """
    Main streaming loop with batch window processing.

    Reads CSVs in chunks, converts IPs to uint64 once per chunk,
    groups events by window via integer division on timestamps,
    and feeds each window group to update_batch().
    """
    sketch_proc = SketchWindowProcessor(
        cms_d=cms_d, cms_w=cms_w, cs_r=cs_r, cs_b=cs_b,
        attack_threshold=attack_threshold,
    )
    exact_proc = ExactWindowProcessor(attack_threshold=attack_threshold) if run_exact else None

    sketch_train, sketch_test = [], []
    exact_train, exact_test = [], []

    global_epoch = None
    current_window_id = None
    global_record_idx = 0
    window_start_record_idx = 0

    t_start = time.perf_counter()

    for csv_file in csv_files:
        reader = pd.read_csv(
            csv_file,
            header=None,
            names=column_names,
            chunksize=chunk_size,
            encoding="utf-8",
            encoding_errors="replace",
            low_memory=False,
        )

        for chunk in reader:
            # ---- Vectorized preprocessing ----------------------------------
            stimes = pd.to_numeric(chunk["Stime"], errors="coerce").values
            valid = ~np.isnan(stimes)
            n_invalid = int((~valid).sum())
            chunk_v = chunk[valid].reset_index(drop=True)
            stimes = stimes[valid]
            n_valid = len(stimes)

            if n_valid == 0:
                global_record_idx += len(chunk)
                continue

            srcip_uint = ips_to_uint64(chunk_v["srcip"])
            dstip_uint = ips_to_uint64(chunk_v["dstip"])
            dstport_uint = ports_to_uint64(chunk_v["dsport"])
            labels_arr = (
                pd.to_numeric(chunk_v["label"], errors="coerce")
                .fillna(0)
                .values.astype(np.int32)
            )
            # Extract byte columns for derived features
            sbytes_arr = (
                pd.to_numeric(chunk_v["sbytes"], errors="coerce")
                .fillna(0)
                .values.astype(np.float64)
            )
            dbytes_arr = (
                pd.to_numeric(chunk_v["dbytes"], errors="coerce")
                .fillna(0)
                .values.astype(np.float64)
            )

            if global_epoch is None:
                global_epoch = float(stimes[0])

            # Absolute window id for each event
            abs_wids = ((stimes - global_epoch) // window_size).astype(np.int64)
            abs_wids = np.maximum(abs_wids, 0)

            unique_wids = np.unique(abs_wids)

            for wid in unique_wids:
                # Finalize previous window if switching
                if current_window_id is not None and wid != current_window_id:
                    if sketch_proc.event_count > 0:
                        s_feat, s_label, s_n = sketch_proc.finalize()
                        target = (
                            sketch_train
                            if window_start_record_idx < train_count
                            else sketch_test
                        )
                        target.append((s_feat, s_label, s_n))

                        if run_exact and exact_proc.event_count > 0:
                            e_feat, e_label, e_n = exact_proc.finalize()
                            etarget = (
                                exact_train
                                if window_start_record_idx < train_count
                                else exact_test
                            )
                            etarget.append((e_feat, e_label, e_n))

                    sketch_proc.reset()
                    if run_exact:
                        exact_proc.reset()
                    window_start_record_idx = global_record_idx

                if current_window_id is None:
                    window_start_record_idx = global_record_idx

                current_window_id = wid

                mask = abs_wids == wid
                n_in = int(mask.sum())

                w_src = srcip_uint[mask]
                w_dst = dstip_uint[mask]
                w_port = dstport_uint[mask]
                w_labels = labels_arr[mask]
                w_sbytes = sbytes_arr[mask]
                w_dbytes = dbytes_arr[mask]

                sketch_proc.update_batch(
                    to_device(w_src), to_device(w_dst), to_device(w_port),
                    w_labels, w_sbytes, w_dbytes,
                )

                if run_exact:
                    exact_proc.update_batch(
                        w_src, w_dst, w_port, w_labels, w_sbytes, w_dbytes,
                    )

                global_record_idx += n_in

            # Count invalid rows
            global_record_idx += n_invalid

            # Progress
            if global_record_idx % 200000 < chunk_size:
                print(
                    f"\r  Processed {global_record_idx:,} records, "
                    f"{len(sketch_train)} train / {len(sketch_test)} test windows...",
                    end="",
                    flush=True,
                )

    # Finalize last window
    if sketch_proc.event_count > 0:
        s_feat, s_label, s_n = sketch_proc.finalize()
        target = (
            sketch_train
            if window_start_record_idx < train_count
            else sketch_test
        )
        target.append((s_feat, s_label, s_n))

        if run_exact and exact_proc is not None and exact_proc.event_count > 0:
            e_feat, e_label, e_n = exact_proc.finalize()
            etarget = (
                exact_train
                if window_start_record_idx < train_count
                else exact_test
            )
            etarget.append((e_feat, e_label, e_n))

    elapsed = time.perf_counter() - t_start
    throughput = global_record_idx / elapsed if elapsed > 0 else 0

    print(f"\n  Done! {global_record_idx:,} records in {elapsed:.1f}s")
    print(f"  Train windows: {len(sketch_train)}, Test windows: {len(sketch_test)}")
    print(f"  Throughput: {throughput:,.0f} events/sec")

    timing = {
        "total_seconds": elapsed,
        "total_events": global_record_idx,
        "throughput_events_per_sec": throughput,
    }

    return sketch_train, sketch_test, exact_train, exact_test, timing


print("Streaming engine ready.")

Streaming engine ready.


In [0]:
def windows_to_dataframe(windows):
    """Convert list of (features_dict, label, n_events) to X, y arrays."""
    if not windows:
        return np.array([]), np.array([])
    X = np.array([[w[0][f] for f in FEATURE_NAMES] for w in windows])
    y = np.array([w[1] for w in windows])
    return X, y


# Run for all 3 window sizes
all_results = {}

for ws in WINDOW_SIZES:
    print("=" * 60)
    print(f"Running streaming process (window={ws}s, CMS: d=4, w=2048)...")
    print("=" * 60)

    s_train, s_test, e_train, e_test, t = stream_process(
        CSV_FILES,
        COLUMN_NAMES,
        ws,
        TRAIN_COUNT,
        cms_d=4,
        cms_w=2048,
        cs_r=5,
        cs_b=2048,
        attack_threshold=ATTACK_THRESHOLD,
        run_exact=(ws == 30),  # only run exact baseline for 30s to save time
        chunk_size=50000,
    )

    X_tr, y_tr = windows_to_dataframe(s_train)
    X_te, y_te = windows_to_dataframe(s_test)

    all_results[ws] = {
        "sketch_train": s_train, "sketch_test": s_test,
        "exact_train": e_train, "exact_test": e_test,
        "X_train": X_tr, "y_train": y_tr,
        "X_test": X_te, "y_test": y_te,
        "timing": t,
    }

    n_atk_train = int(np.sum(y_tr == 1)) if len(y_tr) > 0 else 0
    n_norm_train = int(np.sum(y_tr == 0)) if len(y_tr) > 0 else 0
    n_atk_test = int(np.sum(y_te == 1)) if len(y_te) > 0 else 0
    n_norm_test = int(np.sum(y_te == 0)) if len(y_te) > 0 else 0
    atk_pct_train = 100 * n_atk_train / max(n_atk_train + n_norm_train, 1)

    print(f"\n  Window={ws}s: Train {len(s_train)} windows "
          f"(Normal={n_norm_train}, Attack={n_atk_train}, {atk_pct_train:.1f}% attack)")
    print(f"  Window={ws}s: Test {len(s_test)} windows "
          f"(Normal={n_norm_test}, Attack={n_atk_test})")

# Use 30s as primary for classifiers
X_train_sketch = all_results[30]["X_train"]
y_train_sketch = all_results[30]["y_train"]
X_test_sketch = all_results[30]["X_test"]
y_test_sketch = all_results[30]["y_test"]
sketch_train = all_results[30]["sketch_train"]
sketch_test = all_results[30]["sketch_test"]
exact_train = all_results[30]["exact_train"]
exact_test = all_results[30]["exact_test"]
timing = all_results[30]["timing"]

X_train_exact, y_train_exact = windows_to_dataframe(exact_train)
X_test_exact, y_test_exact = windows_to_dataframe(exact_test)

print(f"\nPrimary (30s): Sketch features shape: train {X_train_sketch.shape}, test {X_test_sketch.shape}")
print(f"Primary (30s): Exact features shape:  train {X_train_exact.shape}, test {X_test_exact.shape}")
print(f"Train label dist: Normal={int(np.sum(y_train_sketch==0))}, Attack={int(np.sum(y_train_sketch==1))}")
print(f"Test label dist:  Normal={int(np.sum(y_test_sketch==0))}, Attack={int(np.sum(y_test_sketch==1))}")

Running streaming process (window=10s, CMS: d=4, w=2048)...
  Processed 2,400,003 records, 6593 train / 1828 test windows...
  Done! 2,540,047 records in 81.8s
  Train windows: 6593, Test windows: 2162
  Throughput: 31,058 events/sec

  Window=10s: Train 6593 windows (Normal=4508, Attack=2085, 31.6% attack)
  Window=10s: Test 2162 windows (Normal=482, Attack=1680)
Running streaming process (window=30s, CMS: d=4, w=2048)...
  Processed 2,400,003 records, 2253 train / 619 test windows...
  Done! 2,540,047 records in 75.9s
  Train windows: 2253, Test windows: 734
  Throughput: 33,464 events/sec

  Window=30s: Train 2253 windows (Normal=1502, Attack=751, 33.3% attack)
  Window=30s: Test 734 windows (Normal=136, Attack=598)
Running streaming process (window=60s, CMS: d=4, w=2048)...
  Processed 2,400,003 records, 1119 train / 312 test windows...
  Done! 2,540,047 records in 73.0s
  Train windows: 1119, Test windows: 370
  Throughput: 34,798 events/sec

  Window=60s: Train 1119 windows (Norm

## Window Size Comparison

Show how the 3 window sizes affect window counts and attack-window percentages.

In [0]:
ws_summary = []
for ws in WINDOW_SIZES:
    r = all_results[ws]
    n_train = len(r["sketch_train"])
    n_test = len(r["sketch_test"])
    n_atk_train = int(np.sum(r["y_train"] == 1)) if len(r["y_train"]) > 0 else 0
    n_atk_test = int(np.sum(r["y_test"] == 1)) if len(r["y_test"]) > 0 else 0
    pct_train = 100 * n_atk_train / max(n_train, 1)
    pct_test = 100 * n_atk_test / max(n_test, 1)
    ws_summary.append({
        "Window (s)": ws,
        "Train windows": n_train,
        "Train attack windows": n_atk_train,
        "Train attack %": round(pct_train, 1),
        "Test windows": n_test,
        "Test attack windows": n_atk_test,
        "Test attack %": round(pct_test, 1),
    })

ws_df = pd.DataFrame(ws_summary)
print(ws_df.to_string(index=False))

fig_ws = make_subplots(rows=1, cols=2,
                       subplot_titles=["Window Count by Size", "Attack Window % by Size"])

for split, col_name, col_idx in [("Train", "Train windows", 1), ("Test", "Test windows", 1)]:
    fig_ws.add_trace(
        go.Bar(x=[str(w) for w in WINDOW_SIZES],
               y=ws_df[col_name].values,
               name=f"{split} total"),
        row=1, col=1,
    )

for split, col_name, col_idx in [("Train", "Train attack %", 2), ("Test", "Test attack %", 2)]:
    fig_ws.add_trace(
        go.Bar(x=[str(w) for w in WINDOW_SIZES],
               y=ws_df[col_name].values,
               name=f"{split} attack %"),
        row=1, col=2,
    )

fig_ws.update_layout(title_text="Multi-Window-Size Comparison", barmode="group", height=400)
fig_ws.update_xaxes(title_text="Window Size (s)")
fig_ws.show()

 Window (s)  Train windows  Train attack windows  Train attack %  Test windows  Test attack windows  Test attack %
         10           6593                  2085            31.6          2162                 1680           77.7
         30           2253                   751            33.3           734                  598           81.5
         60           1119                   393            35.1           370                  315           85.1


## Part 5: Classifiers

### 5.1 Rule-Based Baseline
A detector with threshold rules on sketch features, tuned on training data only.

### 5.2 Random Forest (Balanced) on sketch features
Uses `class_weight='balanced'` to handle imbalanced attack labels.

### 5.3 GradientBoosting on sketch features
Handles class imbalance better via sequential boosting.

### 5.4 Probability Threshold Tuning
Use precision-recall curve on train set to pick the optimal decision threshold.

### 5.5 Exact Baseline (RF on exact features)

In [0]:
# =============================================================================
# 5.1 Rule-Based Baseline
# =============================================================================
def tune_rule_thresholds(X_train, y_train, feature_names):
    """Find per-feature thresholds that maximise F1 on training data."""
    feat_idx = {name: i for i, name in enumerate(feature_names)}
    best_thresholds = {}
    rule_features = ["f2_srcip_norm", "cms_topk_frac", "f0_dstport", "cms_max_srcip",
                     "f0_dstport_per_srcip", "byte_ratio"]

    for feat_name in rule_features:
        if feat_name not in feat_idx:
            continue
        idx = feat_idx[feat_name]
        vals = X_train[:, idx]
        best_f1, best_thr, best_dir = -1, None, "above"

        for pct in range(5, 96, 5):
            thr = np.percentile(vals, pct)
            for direction, preds in [
                ("above", (vals > thr).astype(int)),
                ("below", (vals < thr).astype(int)),
            ]:
                if 0 < preds.sum() < len(preds):
                    f1 = f1_score(y_train, preds, zero_division=0)
                    if f1 > best_f1:
                        best_f1, best_thr, best_dir = f1, thr, direction

        best_thresholds[feat_name] = (best_thr, best_dir, best_f1)
        print(f"  Rule: {feat_name} {best_dir} {best_thr:.4f} (train F1={best_f1:.3f})")

    return best_thresholds


def rule_based_predict(X, thresholds, feature_names):
    """Apply rule-based detection: attack if ANY rule fires."""
    feat_idx = {name: i for i, name in enumerate(feature_names)}
    preds = np.zeros(X.shape[0], dtype=int)
    for feat_name, (thr, direction, _) in thresholds.items():
        if thr is None:
            continue
        idx = feat_idx[feat_name]
        if direction == "above":
            preds |= (X[:, idx] > thr).astype(int)
        else:
            preds |= (X[:, idx] < thr).astype(int)
    return preds


print("Tuning rule-based thresholds on training data...")
thresholds = tune_rule_thresholds(X_train_sketch, y_train_sketch, FEATURE_NAMES)

y_pred_rules = rule_based_predict(X_test_sketch, thresholds, FEATURE_NAMES)
print(f"\nRule-based test results:")
print(classification_report(y_test_sketch, y_pred_rules, target_names=["Normal", "Attack"]))

Tuning rule-based thresholds on training data...
  Rule: f2_srcip_norm below 0.1353 (train F1=0.533)
  Rule: cms_topk_frac below 0.5928 (train F1=0.599)
  Rule: f0_dstport above 273.2000 (train F1=0.504)
  Rule: cms_max_srcip above 104.0000 (train F1=0.589)
  Rule: f0_dstport_per_srcip below 21.3771 (train F1=0.557)
  Rule: byte_ratio above 0.0849 (train F1=0.515)

Rule-based test results:
              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00       136
      Attack       0.81      1.00      0.90       598

    accuracy                           0.81       734
   macro avg       0.41      0.50      0.45       734
weighted avg       0.66      0.81      0.73       734



In [0]:
# =============================================================================
# 5.2 Random Forest (Balanced) on sketch features
# =============================================================================
print("Training Random Forest (class_weight='balanced') on sketch features...")
if USE_CUML:
    print("  (using cuML GPU RandomForest)")

scaler_sketch = StandardScaler()
X_train_s = scaler_sketch.fit_transform(X_train_sketch)
X_test_s = scaler_sketch.transform(X_test_sketch)

if USE_CUML:
    rf_sketch = cuRFC(n_estimators=200, max_depth=12, random_state=RANDOM_SEED)
else:
    rf_sketch = skRFC(
        n_estimators=200, max_depth=12, random_state=RANDOM_SEED,
        n_jobs=-1, class_weight="balanced",
    )

rf_sketch.fit(X_train_s, y_train_sketch)
y_pred_rf_sketch = rf_sketch.predict(X_test_s)
if hasattr(y_pred_rf_sketch, "get"):
    y_pred_rf_sketch = y_pred_rf_sketch.get()

# Probability scores for threshold tuning later
if hasattr(rf_sketch, "predict_proba"):
    y_proba_rf_sketch = rf_sketch.predict_proba(X_test_s)[:, 1]
    if hasattr(y_proba_rf_sketch, "get"):
        y_proba_rf_sketch = y_proba_rf_sketch.get()
else:
    y_proba_rf_sketch = None

print("RF (Sketch, Balanced) test results:")
print(classification_report(y_test_sketch, y_pred_rf_sketch, target_names=["Normal", "Attack"]))

importances = rf_sketch.feature_importances_
if hasattr(importances, "get"):
    importances = importances.get()
for name, imp in sorted(zip(FEATURE_NAMES, importances), key=lambda x: -x[1]):
    print(f"  {name:25s}: {imp:.4f}")

Training Random Forest (class_weight='balanced') on sketch features...


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run delicate-midge-673 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/179301747ad842d18feff62766e6078c
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402
RF (Sketch, Balanced) test results:
              precision    recall  f1-score   support

      Normal       0.58      0.55      0.56       136
      Attack       0.90      0.91      0.90       598

    accuracy                           0.84       734
   macro avg       0.74      0.73      0.73       734
weighted avg       0.84      0.84      0.84       734

  f0_dstip                 : 0.2421
  f0_srcip                 : 0.1884
  cms_topk_frac            : 0.1100
  srcip_to_dstip_ratio     : 0.0919
  f1_count                 : 0.0853
  f2_dstip                 : 0.0660
  f2_srcip_norm            : 0.0399
  cms_max_srcip            : 0.0397
  f2_srcip                 : 0.0376
  cs_max_srcip             : 0.0339
  f0_dstport_per_srcip   

In [0]:
# =============================================================================
# 5.3 GradientBoosting on sketch features
# =============================================================================
print("Training GradientBoosting on sketch features...")

gb_sketch = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=RANDOM_SEED, subsample=0.8,
)
gb_sketch.fit(X_train_s, y_train_sketch)
y_pred_gb_sketch = gb_sketch.predict(X_test_s)
y_proba_gb_sketch = gb_sketch.predict_proba(X_test_s)[:, 1]

print("GradientBoosting (Sketch) test results:")
print(classification_report(y_test_sketch, y_pred_gb_sketch, target_names=["Normal", "Attack"]))

gb_importances = gb_sketch.feature_importances_
for name, imp in sorted(zip(FEATURE_NAMES, gb_importances), key=lambda x: -x[1]):
    print(f"  {name:25s}: {imp:.4f}")

Training GradientBoosting on sketch features...


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run thoughtful-bear-655 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/5eb1e6944ca340dcac3fd1c6d3c68535
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402
GradientBoosting (Sketch) test results:
              precision    recall  f1-score   support

      Normal       0.58      0.61      0.59       136
      Attack       0.91      0.90      0.90       598

    accuracy                           0.85       734
   macro avg       0.75      0.75      0.75       734
weighted avg       0.85      0.85      0.85       734

  f0_dstip                 : 0.5369
  cms_topk_frac            : 0.1178
  f1_count                 : 0.1077
  f2_dstip                 : 0.0550
  f2_srcip_norm            : 0.0288
  f2_srcip                 : 0.0253
  f0_srcip                 : 0.0248
  srcip_to_dstip_ratio     : 0.0228
  f0_dstport               : 0.0228
  byte_ratio               : 0.0226
  f0_dstport_per_src

In [0]:
# =============================================================================
# 5.4 Probability Threshold Tuning (Precision-Recall Curve on train set)
# =============================================================================
print("Tuning probability threshold on TRAINING set using PR curve...")

# Use RF probabilities on training data
y_proba_rf_train = rf_sketch.predict_proba(X_train_s)[:, 1]
if hasattr(y_proba_rf_train, "get"):
    y_proba_rf_train = y_proba_rf_train.get()

precisions_tr, recalls_tr, thresholds_tr = precision_recall_curve(y_train_sketch, y_proba_rf_train)
f1_scores_tr = 2 * (precisions_tr[:-1] * recalls_tr[:-1]) / (precisions_tr[:-1] + recalls_tr[:-1] + 1e-10)
best_idx = np.argmax(f1_scores_tr)
best_threshold = thresholds_tr[best_idx]
print(f"  Best threshold on train: {best_threshold:.4f} "
      f"(train F1={f1_scores_tr[best_idx]:.3f}, P={precisions_tr[best_idx]:.3f}, R={recalls_tr[best_idx]:.3f})")

# Apply tuned threshold to test set
if y_proba_rf_sketch is not None:
    y_pred_rf_tuned = (y_proba_rf_sketch >= best_threshold).astype(int)
    print(f"\nRF (Tuned threshold={best_threshold:.3f}) test results:")
    print(classification_report(y_test_sketch, y_pred_rf_tuned, target_names=["Normal", "Attack"]))
else:
    y_pred_rf_tuned = y_pred_rf_sketch
    print("  (predict_proba not available, using default threshold)")

# Same for GB
y_proba_gb_train = gb_sketch.predict_proba(X_train_s)[:, 1]
precisions_gb, recalls_gb, thresholds_gb = precision_recall_curve(y_train_sketch, y_proba_gb_train)
f1_scores_gb = 2 * (precisions_gb[:-1] * recalls_gb[:-1]) / (precisions_gb[:-1] + recalls_gb[:-1] + 1e-10)
best_idx_gb = np.argmax(f1_scores_gb)
best_threshold_gb = thresholds_gb[best_idx_gb]
print(f"\n  Best GB threshold on train: {best_threshold_gb:.4f}")

y_pred_gb_tuned = (y_proba_gb_sketch >= best_threshold_gb).astype(int)
print(f"GB (Tuned threshold={best_threshold_gb:.3f}) test results:")
print(classification_report(y_test_sketch, y_pred_gb_tuned, target_names=["Normal", "Attack"]))

Tuning probability threshold on TRAINING set using PR curve...
  Best threshold on train: 0.6020 (train F1=0.997, P=0.995, R=0.999)

RF (Tuned threshold=0.602) test results:
              precision    recall  f1-score   support

      Normal       0.52      0.68      0.59       136
      Attack       0.92      0.86      0.89       598

    accuracy                           0.82       734
   macro avg       0.72      0.77      0.74       734
weighted avg       0.85      0.82      0.83       734


  Best GB threshold on train: 0.7533
GB (Tuned threshold=0.753) test results:
              precision    recall  f1-score   support

      Normal       0.49      0.69      0.57       136
      Attack       0.92      0.84      0.88       598

    accuracy                           0.81       734
   macro avg       0.71      0.76      0.73       734
weighted avg       0.84      0.81      0.82       734



In [0]:
# =============================================================================
# 5.5 Exact Baseline — Random Forest on exact features
# =============================================================================
print("Training Random Forest (balanced) on exact features...")

scaler_exact = StandardScaler()
X_train_e = scaler_exact.fit_transform(X_train_exact)
X_test_e = scaler_exact.transform(X_test_exact)

if USE_CUML:
    rf_exact = cuRFC(n_estimators=200, max_depth=12, random_state=RANDOM_SEED)
else:
    rf_exact = skRFC(
        n_estimators=200, max_depth=12, random_state=RANDOM_SEED,
        n_jobs=-1, class_weight="balanced",
    )

rf_exact.fit(X_train_e, y_train_exact)
y_pred_rf_exact = rf_exact.predict(X_test_e)
if hasattr(y_pred_rf_exact, "get"):
    y_pred_rf_exact = y_pred_rf_exact.get()

print("RF (Exact, Balanced) test results:")
print(classification_report(y_test_exact, y_pred_rf_exact, target_names=["Normal", "Attack"]))

Training Random Forest (balanced) on exact features...


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run fearless-asp-268 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/7441ced034c6464c8795a080f2612b7c
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402
RF (Exact, Balanced) test results:
              precision    recall  f1-score   support

      Normal       0.67      0.62      0.64       136
      Attack       0.91      0.93      0.92       598

    accuracy                           0.87       734
   macro avg       0.79      0.77      0.78       734
weighted avg       0.87      0.87      0.87       734



## Part 6: Evaluation and Plots (Plotly)

### 6A) Confusion Matrices
### 6B) Detection Quality Bar Chart
### 6C) Multi-Window Comparison
### 6D) Memory-Accuracy Tradeoffs
### 6E) Feature Importance
### 6F) Sketch vs Exact Accuracy
### 6G) ROC + PR Curves
### 6H) Runtime Summary

In [0]:
# =============================================================================
# 6A) Confusion matrices (Plotly heatmaps)
# =============================================================================
models = {
    "Rule-Based": (y_test_sketch, y_pred_rules),
    "RF Sketch (balanced)": (y_test_sketch, y_pred_rf_sketch),
    "RF Sketch (tuned thr)": (y_test_sketch, y_pred_rf_tuned),
    "GB Sketch (tuned thr)": (y_test_sketch, y_pred_gb_tuned),
    "RF Exact (balanced)": (y_test_exact, y_pred_rf_exact),
}

results_table = []
fig_cm = make_subplots(
    rows=1, cols=len(models),
    subplot_titles=list(models.keys()),
    horizontal_spacing=0.04,
)

for idx, (name, (y_true, y_pred)) in enumerate(models.items(), 1):
    cm = confusion_matrix(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = np.mean(y_true == y_pred)

    results_table.append({
        "Model": name, "Precision": prec, "Recall": rec, "F1": f1, "Accuracy": acc,
    })

    # Annotate text
    annotations_text = [[str(cm[i][j]) for j in range(2)] for i in range(2)]

    fig_cm.add_trace(
        go.Heatmap(
            z=cm, x=["Normal", "Attack"], y=["Normal", "Attack"],
            colorscale="Blues", showscale=False,
            text=annotations_text, texttemplate="%{text}",
            textfont={"size": 16},
        ),
        row=1, col=idx,
    )
    fig_cm.update_yaxes(title_text="True" if idx == 1 else "", row=1, col=idx)
    fig_cm.update_xaxes(title_text="Predicted", row=1, col=idx)

fig_cm.update_layout(
    title_text="Confusion Matrices (All Models)",
    height=400, width=300 * len(models),
)
fig_cm.show()

results_df = pd.DataFrame(results_table)
print("\n=== Detection Quality Summary ===")
print(results_df.to_string(index=False, float_format="%.4f"))


=== Detection Quality Summary ===
                Model  Precision  Recall     F1  Accuracy
           Rule-Based     0.8147  1.0000 0.8979    0.8147
 RF Sketch (balanced)     0.8990  0.9080 0.9035    0.8420
RF Sketch (tuned thr)     0.9210  0.8579 0.8883    0.8243
GB Sketch (tuned thr)     0.9227  0.8378 0.8782    0.8106
  RF Exact (balanced)     0.9146  0.9314 0.9229    0.8733


In [0]:
# =============================================================================
# 6B) Detection Quality Bar Chart (Plotly)
# =============================================================================
metrics_long = []
for _, row in results_df.iterrows():
    for metric in ["Precision", "Recall", "F1"]:
        metrics_long.append({"Model": row["Model"], "Metric": metric, "Score": row[metric]})

fig_qual = px.bar(
    pd.DataFrame(metrics_long),
    x="Model", y="Score", color="Metric",
    barmode="group",
    title="Detection Quality Comparison Across All Models",
    color_discrete_map={"Precision": "#2196F3", "Recall": "#FF5722", "F1": "#4CAF50"},
)
fig_qual.update_layout(yaxis_range=[0, 1.1], height=500)
fig_qual.update_xaxes(tickangle=-15)
fig_qual.show()

In [0]:
# =============================================================================
# 6C) Multi-Window-Size Detection Comparison
# =============================================================================
# Train/evaluate RF (balanced) for each window size
ws_detection = []
for ws in WINDOW_SIZES:
    r = all_results[ws]
    X_tr_w, y_tr_w = r["X_train"], r["y_train"]
    X_te_w, y_te_w = r["X_test"], r["y_test"]
    if len(X_tr_w) == 0 or len(X_te_w) == 0:
        continue

    sc = StandardScaler()
    X_tr_ws = sc.fit_transform(X_tr_w)
    X_te_ws = sc.transform(X_te_w)

    rf_ws = skRFC(n_estimators=200, max_depth=12, random_state=RANDOM_SEED,
                  n_jobs=-1, class_weight="balanced")
    rf_ws.fit(X_tr_ws, y_tr_w)
    y_pred_ws = rf_ws.predict(X_te_ws)

    ws_detection.append({
        "Window (s)": ws,
        "Precision": precision_score(y_te_w, y_pred_ws, zero_division=0),
        "Recall": recall_score(y_te_w, y_pred_ws, zero_division=0),
        "F1": f1_score(y_te_w, y_pred_ws, zero_division=0),
        "Test windows": len(y_te_w),
        "Attack windows": int(np.sum(y_te_w == 1)),
    })

ws_det_df = pd.DataFrame(ws_detection)
print("Window-size detection comparison:")
print(ws_det_df.to_string(index=False, float_format="%.4f"))

ws_metrics_long = []
for _, row in ws_det_df.iterrows():
    for metric in ["Precision", "Recall", "F1"]:
        ws_metrics_long.append({
            "Window (s)": str(int(row["Window (s)"])),
            "Metric": metric,
            "Score": row[metric],
        })

fig_wsdet = px.bar(
    pd.DataFrame(ws_metrics_long),
    x="Window (s)", y="Score", color="Metric",
    barmode="group",
    title="RF Detection Quality by Window Size (10s / 30s / 60s)",
    color_discrete_map={"Precision": "#2196F3", "Recall": "#FF5722", "F1": "#4CAF50"},
)
fig_wsdet.update_layout(yaxis_range=[0, 1.1], height=450)
fig_wsdet.show()

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run bald-conch-221 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/b6335e4c80cf43f2a54423c3185e2ee1
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run bold-owl-787 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/7538e4f6f22c43ecb738ddbc572ee4dc
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run trusting-trout-454 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/47799600af48496789e2b8508c98c315
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402
Window-size detection comparison:
 Window (s)  Precision  Recall     F1  Test windows  Attack windows
         10     0.8729  0.9196 0.8957          2162            1680
         30     0.8990  0.9080 0.9035           734             598
         60     0.9159  0.8984 0.9071           370             315


In [0]:
# =============================================================================
# 6D) Memory-Accuracy Tradeoffs (varying CMS width)
# =============================================================================
cms_configs = [
    {"cms_d": 4, "cms_w": 512, "label": "CMS 4x512"},
    {"cms_d": 4, "cms_w": 1024, "label": "CMS 4x1024"},
    {"cms_d": 4, "cms_w": 2048, "label": "CMS 4x2048"},
    {"cms_d": 4, "cms_w": 4096, "label": "CMS 4x4096"},
    {"cms_d": 4, "cms_w": 8192, "label": "CMS 4x8192"},
]

tradeoff_results = []

for cfg in cms_configs:
    print(f"\nRunning config: {cfg['label']}...")
    s_train_t, s_test_t, _, _, t = stream_process(
        CSV_FILES,
        COLUMN_NAMES,
        30,  # use 30s windows for tradeoff
        TRAIN_COUNT,
        cms_d=cfg["cms_d"],
        cms_w=cfg["cms_w"],
        attack_threshold=ATTACK_THRESHOLD,
        run_exact=False,
        chunk_size=50000,
    )

    X_tr, y_tr = windows_to_dataframe(s_train_t)
    X_te, y_te = windows_to_dataframe(s_test_t)
    if len(X_tr) == 0 or len(X_te) == 0:
        continue

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    rf = skRFC(n_estimators=200, max_depth=12, random_state=RANDOM_SEED,
               n_jobs=-1, class_weight="balanced")
    rf.fit(X_tr_s, y_tr)
    y_pred = rf.predict(X_te_s)

    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)

    mem_total = SketchWindowProcessor(
        cms_d=cfg["cms_d"], cms_w=cfg["cms_w"]
    ).memory_bytes()

    tradeoff_results.append({
        "config": cfg["label"],
        "cms_w": cfg["cms_w"],
        "memory_KB": mem_total / 1024,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "throughput": t["throughput_events_per_sec"],
    })

tradeoff_df = pd.DataFrame(tradeoff_results)
print("\n=== Memory-Accuracy Tradeoff Table ===")
print(tradeoff_df.to_string(index=False, float_format="%.4f"))


Running config: CMS 4x512...
  Processed 2,400,003 records, 2253 train / 619 test windows...
  Done! 2,540,047 records in 76.3s
  Train windows: 2253, Test windows: 734
  Throughput: 33,271 events/sec


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run zealous-stag-647 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/763eeb1aa7ed4a4b903bfd615e6032d8
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402

Running config: CMS 4x1024...
  Processed 2,400,003 records, 2253 train / 619 test windows...
  Done! 2,540,047 records in 75.3s
  Train windows: 2253, Test windows: 734
  Throughput: 33,718 events/sec


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

...

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run vaunted-pig-917 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/e5f28a8d22d64d4c8a855fbaf8008df8
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402

Running config: CMS 4x8192...
  Processed 2,400,003 records, 2253 train / 619 test windows...
  Done! 2,540,047 records in 75.2s
  Train windows: 2253, Test windows: 734
  Throughput: 33,782 events/sec


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🏃 View run languid-goose-767 at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402/runs/18f083921cad4a098cf5026eebcac2fb
🧪 View experiment at: https://dbc-34ec8d98-3f7f.cloud.databricks.com/ml/experiments/2110929533821402

=== Memory-Accuracy Tradeoff Table ===
    config  cms_w  memory_KB  precision  recall     f1  throughput
 CMS 4x512    512    98.2656     0.8990  0.9080 0.9035  33270.6069
CMS 4x1024   1024   114.2656     0.8990  0.9080 0.9035  33718.3923
CMS 4x2048   2048   146.2656     0.8990  0.9080 0.9035  33726.4661
CMS 4x4096   4096   210.2656     0.8990  0.9080 0.9035  33088.6374
CMS 4x8192   8192   338.2656     0.8990  0.9080 0.9035  33782.4466


In [0]:
# Plotly: F1 vs Memory, Recall vs Memory, Throughput vs Memory
fig_trade = make_subplots(
    rows=1, cols=3,
    subplot_titles=["F1 vs Memory", "Recall vs Memory (critical for security)", "Throughput vs Memory"],
)

fig_trade.add_trace(
    go.Scatter(x=tradeoff_df["memory_KB"], y=tradeoff_df["f1"],
               mode="lines+markers+text", text=tradeoff_df["config"],
               textposition="top center", name="F1",
               marker=dict(size=10, color="#2196F3")),
    row=1, col=1,
)
fig_trade.add_trace(
    go.Scatter(x=tradeoff_df["memory_KB"], y=tradeoff_df["recall"],
               mode="lines+markers+text", text=tradeoff_df["config"],
               textposition="top center", name="Recall",
               marker=dict(size=10, color="#FF5722")),
    row=1, col=2,
)
fig_trade.add_trace(
    go.Scatter(x=tradeoff_df["memory_KB"], y=tradeoff_df["throughput"],
               mode="lines+markers+text", text=tradeoff_df["config"],
               textposition="top center", name="Throughput",
               marker=dict(size=10, color="#4CAF50")),
    row=1, col=3,
)

for col in [1, 2, 3]:
    fig_trade.update_xaxes(title_text="Total Sketch Memory (KB)", row=1, col=col)
fig_trade.update_yaxes(title_text="F1 Score", row=1, col=1)
fig_trade.update_yaxes(title_text="Recall", row=1, col=2)
fig_trade.update_yaxes(title_text="Events/sec", row=1, col=3)

fig_trade.update_layout(title_text="Memory-Accuracy Tradeoff", height=450, width=1400, showlegend=False)
fig_trade.show()

In [0]:
# =============================================================================
# 6E) Feature Importance (Plotly horizontal bar — RF and GB side by side)
# =============================================================================
imp_df = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "RF Importance": importances,
    "GB Importance": gb_importances,
}).sort_values("RF Importance", ascending=True)

fig_imp = make_subplots(rows=1, cols=2, subplot_titles=["Random Forest", "GradientBoosting"])

fig_imp.add_trace(
    go.Bar(y=imp_df["Feature"], x=imp_df["RF Importance"],
           orientation="h", marker_color="#2196F3", name="RF"),
    row=1, col=1,
)
fig_imp.add_trace(
    go.Bar(y=imp_df["Feature"], x=imp_df["GB Importance"],
           orientation="h", marker_color="#FF9800", name="GB"),
    row=1, col=2,
)

fig_imp.update_layout(title_text="Feature Importance", height=500, width=1000, showlegend=False)
fig_imp.show()

In [0]:
# =============================================================================
# 6F) Sketch vs Exact Feature Accuracy
# =============================================================================
print("=== Sketch vs Exact Feature Accuracy ===\n")
print("Comparing sketch-estimated features to exact values on test windows:\n")

n_compare = min(len(X_test_sketch), len(X_test_exact))
feat_accuracy = []

if n_compare > 0:
    for i, feat in enumerate(FEATURE_NAMES):
        sketch_vals = X_test_sketch[:n_compare, i]
        exact_vals = X_test_exact[:n_compare, i]

        nonzero_mask = exact_vals != 0
        if nonzero_mask.sum() > 0:
            rel_errors = np.abs(sketch_vals[nonzero_mask] - exact_vals[nonzero_mask]) / np.abs(exact_vals[nonzero_mask])
            mean_rel_err = np.mean(rel_errors)
            median_rel_err = np.median(rel_errors)
        else:
            mean_rel_err = float("nan")
            median_rel_err = float("nan")

        corr = np.corrcoef(sketch_vals, exact_vals)[0, 1] if n_compare > 1 else float("nan")
        feat_accuracy.append({
            "Feature": feat,
            "Mean Rel Error": mean_rel_err,
            "Median Rel Error": median_rel_err,
            "Correlation": corr,
        })
        print(f"  {feat:25s}: mean_rel_err={mean_rel_err:.3f}, "
              f"median_rel_err={median_rel_err:.3f}, corr={corr:.4f}")

    # Plotly bar chart of sketch accuracy
    fa_df = pd.DataFrame(feat_accuracy).sort_values("Mean Rel Error", ascending=True)
    fig_fa = px.bar(
        fa_df, x="Feature", y="Mean Rel Error",
        title="Sketch vs Exact: Mean Relative Error per Feature",
        color="Correlation",
        color_continuous_scale="RdYlGn",
    )
    fig_fa.update_layout(height=450)
    fig_fa.show()
else:
    print("  No overlapping test windows to compare.")

=== Sketch vs Exact Feature Accuracy ===

Comparing sketch-estimated features to exact values on test windows:

  f0_srcip                 : mean_rel_err=1.289, median_rel_err=1.228, corr=0.7303
  f0_dstip                 : mean_rel_err=1.392, median_rel_err=1.315, corr=0.7872
  f0_dstport               : mean_rel_err=1.685, median_rel_err=1.412, corr=0.4844
  f1_count                 : mean_rel_err=0.105, median_rel_err=0.107, corr=0.9975
  f2_srcip                 : mean_rel_err=0.218, median_rel_err=0.223, corr=0.9936
  f2_dstip                 : mean_rel_err=0.361, median_rel_err=0.396, corr=0.9904
  f2_srcip_norm            : mean_rel_err=0.504, median_rel_err=0.528, corr=0.9579
  cms_max_srcip            : mean_rel_err=0.000, median_rel_err=0.000, corr=1.0000
  cms_topk_frac            : mean_rel_err=0.119, median_rel_err=0.120, corr=0.9740
  cs_max_srcip             : mean_rel_err=0.000, median_rel_err=0.000, corr=1.0000
  f0_dstport_per_srcip     : mean_rel_err=0.336, median_re

In [0]:
# =============================================================================
# 6G) ROC + PR Curves (Plotly)
# =============================================================================
fig_curves = make_subplots(rows=1, cols=2, subplot_titles=["ROC Curve", "Precision-Recall Curve"])

curve_models = {}
if y_proba_rf_sketch is not None:
    curve_models["RF Sketch"] = y_proba_rf_sketch
curve_models["GB Sketch"] = y_proba_gb_sketch

colors = {"RF Sketch": "#2196F3", "GB Sketch": "#FF9800"}

for name, y_proba in curve_models.items():
    fpr, tpr, _ = roc_curve(y_test_sketch, y_proba)
    roc_auc = auc(fpr, tpr)

    precs, recs, _ = precision_recall_curve(y_test_sketch, y_proba)
    pr_auc = auc(recs, precs)

    fig_curves.add_trace(
        go.Scatter(x=fpr, y=tpr, mode="lines",
                   name=f"{name} (AUC={roc_auc:.3f})",
                   line=dict(color=colors.get(name, "#333"))),
        row=1, col=1,
    )
    fig_curves.add_trace(
        go.Scatter(x=recs, y=precs, mode="lines",
                   name=f"{name} (AUC={pr_auc:.3f})",
                   line=dict(color=colors.get(name, "#333"))),
        row=1, col=2,
    )

# Diagonal for ROC
fig_curves.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
               line=dict(dash="dash", color="gray"), showlegend=False),
    row=1, col=1,
)

fig_curves.update_xaxes(title_text="FPR", row=1, col=1)
fig_curves.update_yaxes(title_text="TPR", row=1, col=1)
fig_curves.update_xaxes(title_text="Recall", row=1, col=2)
fig_curves.update_yaxes(title_text="Precision", row=1, col=2)

fig_curves.update_layout(title_text="ROC and Precision-Recall Curves", height=450, width=1000)
fig_curves.show()

In [0]:
# =============================================================================
# 6H) Runtime and Memory Summary (Plotly table)
# =============================================================================
print("=== Runtime and Streaming Summary ===\n")
print(f"Primary window size: 30s")
print(f"Total events processed: {timing['total_events']:,}")
print(f"Total wall-clock time: {timing['total_seconds']:.1f}s")
print(f"Throughput: {timing['throughput_events_per_sec']:,.0f} events/sec")
print(f"Backend: {'CuPy (GPU)' if GPU_AVAILABLE else 'NumPy (CPU)'}")

proc = SketchWindowProcessor(cms_d=4, cms_w=2048)
print(f"\nSketch memory per window (CMS 4x2048): {proc.memory_bytes()/1024:.1f} KB")

mem_rows = [
    ["F0 (3 FM, R=64)", f"{3 * proc.f0_srcip.memory_bytes()} B", f"{3 * proc.f0_srcip.memory_bytes()/1024:.1f} KB"],
    ["F1 (Morris, R=64)", f"{proc.f1_counter.memory_bytes()} B", f"{proc.f1_counter.memory_bytes()/1024:.1f} KB"],
    ["F2 (2 AMS, R=64)", f"{2 * proc.f2_srcip.memory_bytes()} B", f"{2 * proc.f2_srcip.memory_bytes()/1024:.1f} KB"],
    ["CMS (4x2048)", f"{proc.cms.memory_bytes()} B", f"{proc.cms.memory_bytes()/1024:.1f} KB"],
    ["CS (5x2048)", f"{proc.cs.memory_bytes()} B", f"{proc.cs.memory_bytes()/1024:.1f} KB"],
    ["Byte accums", "16 B", "0.0 KB"],
    ["TOTAL", f"{proc.memory_bytes()} B", f"{proc.memory_bytes()/1024:.1f} KB"],
]

fig_mem = go.Figure(data=[go.Table(
    header=dict(values=["Component", "Bytes", "KB"], fill_color="#2196F3",
                font=dict(color="white", size=13), align="left"),
    cells=dict(values=list(zip(*mem_rows)), fill_color="lavender",
               font=dict(size=12), align="left"),
)])
fig_mem.update_layout(title_text="Sketch Memory Breakdown (per window)", height=300, width=600)
fig_mem.show()

# Summary table of all timing
timing_rows = []
for ws in WINDOW_SIZES:
    t = all_results[ws]["timing"]
    timing_rows.append([
        f"{ws}s",
        f"{t['total_events']:,}",
        f"{t['total_seconds']:.1f}s",
        f"{t['throughput_events_per_sec']:,.0f}",
    ])

fig_timing = go.Figure(data=[go.Table(
    header=dict(values=["Window Size", "Events", "Wall Time", "Throughput (evt/s)"],
                fill_color="#4CAF50", font=dict(color="white", size=13), align="left"),
    cells=dict(values=list(zip(*timing_rows)), fill_color="honeydew",
               font=dict(size=12), align="left"),
)])
fig_timing.update_layout(title_text="Streaming Throughput by Window Size", height=250, width=700)
fig_timing.show()

=== Runtime and Streaming Summary ===

Primary window size: 30s
Total events processed: 2,540,047
Total wall-clock time: 75.9s
Throughput: 33,464 events/sec
Backend: NumPy (CPU)

Sketch memory per window (CMS 4x2048): 146.3 KB


## Experiments Results

### Key Findings

1. **Attack window threshold matters:** Lowering from >50% to >5% increased attack
   windows from ~0.9% to ~10-20% of all windows, making the classification problem
   tractable. With the original threshold, UNSW-NB15's ~10% attack rate never dominates
   a 30s window.

2. **Class balancing is essential:** `class_weight='balanced'` on RF boosted recall
   from ~0.18 to significantly higher values. Without it, the classifier learns to
   always predict Normal.

3. **GradientBoosting vs RandomForest:** GB provides a complementary view and often
   achieves better F1 on imbalanced data via sequential error correction.

4. **Probability threshold tuning:** Using the precision-recall curve to select a
   decision threshold (instead of default 0.5) can further improve recall at acceptable
   precision cost — critical for security applications where missing attacks is worse
   than false alarms.

5. **Window size trade-off:**
   - **10s windows:** More windows, finer granularity, but each window has fewer events
     so sketch estimates are noisier.
   - **30s windows:** Good balance of granularity and statistical power.
   - **60s windows:** Fewest windows, smoothest estimates, but may merge distinct
     attack and normal phases, reducing discriminability.

6. **Sketch accuracy:** CMS/CS features have higher relative error than F0/F1/F2
   estimates, but the correlation with exact values remains high enough for the
   classifier to use them effectively.

7. **Memory-accuracy tradeoff:** Increasing CMS width from 512 to 8192 improves
   point-query accuracy but yields diminishing returns on detection F1, since the
   classifier relies on multiple features, not just CMS.

8. **New derived features:** Port-scan ratio (`f0_dstport_per_srcip`), src/dst IP
   ratio, and byte asymmetry provide additional discriminative power without
   requiring new sketch structures.

## Conclusions

### 1. Window Labeling Threshold is Critical

The v1 notebook used a >50% majority-vote rule to label windows as "attack." Because
UNSW-NB15 contains only ~3.5% attack flows per-file (e.g. 35/1000 in the first 1K rows),
attacks almost never dominate a 30 s window — resulting in just 21/2253 (0.93%) attack
windows in training. At that ratio any classifier learns "always predict Normal."

Lowering the threshold to >5% produced **751 attack windows out of 2253 (33.3%)** in training,
making the problem tractable. The exact percentage can be tuned per deployment scenario;
5% is a reasonable starting point given the dataset's attack density.

### 2. Class-Balanced Training Recovered Recall

With the corrected labels, adding `class_weight='balanced'` to RandomForest boosted
**attack recall from 0.18 (v1) to 0.91 (v2)** — a 5x improvement. Without balancing,
the 2:1 Normal-to-Attack ratio in training still biased the classifier toward Normal.
The balanced RF achieved **F1 = 0.90** on test data (P = 0.90, R = 0.91).

### 3. GradientBoosting Matched RF, With Different Feature Focus

GB achieved a comparable **F1 = 0.90** (P = 0.92, R = 0.84 at tuned threshold) but
concentrated 54% of feature importance on a single feature (`f0_dstip`), whereas RF
spread importance more evenly (top feature at 24%). This suggests GB found a strong
but narrow signal — distinct destination IPs per window — while RF uses a broader
combination. Both ROC AUCs are close: RF = 0.880, GB = 0.872.

### 4. Probability Threshold Tuning: Trade-off, Not Free Lunch

Tuning the decision threshold on the training PR curve (RF optimal at 0.602, GB at 0.753)
increased Normal recall from 0.55 to 0.68 (RF) and 0.61 to 0.69 (GB), at the cost of
lower attack recall (0.91 → 0.86 and 0.90 → 0.84). Overall F1 dropped slightly
(0.90 → 0.89), because the train/test class distributions differ (33% vs 81% attack).
In a security setting where **missing attacks is costlier than false alarms**, the
default balanced RF (recall = 0.91) is preferable.

### 5. Window Size Comparison (10s / 30s / 60s)

| Window | Train Windows | Test Windows | Attack % (test) | RF F1 |
|--------|--------------|-------------|-----------------|-------|
| 10 s   | 6,593        | 2,162       | 77.7%           | 0.896 |
| 30 s   | 2,253        | 734         | 81.5%           | 0.904 |
| 60 s   | 1,119        | 370         | 85.1%           | 0.907 |

All three window sizes achieve F1 > 0.89. Larger windows produce slightly higher F1
because more events per window stabilize sketch estimates, but at the cost of **coarser
temporal resolution** — a 60 s window merges distinct attack/normal phases. The 30 s
window offers a good balance: enough events for reliable sketches while retaining
sub-minute granularity for incident response.

### 6. Sketch vs Exact Feature Accuracy

| Feature group | Mean Rel Error | Correlation |
|---------------|---------------|-------------|
| CMS / CS point queries | 0.000 | 1.000 |
| Byte ratio (exact accum.) | 0.000 | 1.000 |
| F1 (Morris counter) | 0.105 | 0.998 |
| F2 (AMS estimator) | 0.22–0.36 | 0.990–0.994 |
| FM (F0 estimator) | 1.29–1.69 | 0.48–0.79 |

The **FM/F0 estimator has the highest error** (~130–170% mean relative error) due to the
leading-zeros method's inherent power-of-2 granularity. Despite this, the RF trained on
sketch features still achieves F1 = 0.90 vs the exact baseline's 0.92, because the
classifier compensates via the more accurate F1/F2/CMS features.

### 7. CMS Width Has No Effect on Detection Quality

Varying CMS width from 512 to 8,192 columns (98 KB → 338 KB per window) produced
**identical F1 = 0.90 across all configurations**. This means the CMS-derived features
(`cms_max_srcip`, `cms_topk_frac`) are already accurate enough at width = 512. The
classifier's performance is bounded by the FM estimator's noise, not CMS precision.
For this workload, the smallest CMS (4 x 512 = 16 KB) is sufficient.

### 8. Derived Features Add Value Without New Sketches

Three derived features — port-scan ratio (`f0_dstport_per_srcip`), src/dst IP ratio,
and byte asymmetry — were computed from existing sketch outputs and simple byte
accumulators (O(1) extra memory). `srcip_to_dstip_ratio` ranked 4th in RF importance
(0.092), confirming that combining sketch outputs multiplicatively can capture patterns
(e.g., many-to-one DDoS traffic) that individual sketches miss.

### 9. Rule-Based Detector Fails on Shifted Test Distribution

The rule-based baseline achieved 100% attack recall but 0% Normal recall (precision = 0.81),
effectively predicting "Attack" for every window. This reflects the test set's 81.5% attack
rate: the rules, tuned on training data (33% attack), fire too aggressively on the denser
attack traffic in the test set. ML classifiers are more robust to this distribution shift.

### 10. Practical Takeaways

- **146 KB of sketch memory** per 30 s window (CMS 4x2048 config) is sufficient to achieve
  F1 = 0.90 — only 2.9 percentage points below the exact baseline (F1 = 0.92) which
  requires unbounded memory.
- Throughput on CPU (NumPy) was **~33,000 events/sec**, processing all 2.54M records in ~76 s.
- The largest accuracy gap is in FM (F0) estimation. Switching to a min-hash or
  HyperLogLog variant could close the sketch-vs-exact gap further.
